# 03 · Validate PostgreSQL and the dataset tables

In [1]:
import os
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host=os.environ.get("DATASET_POSTGRES_HOST", "dataset-postgres"),
    port=os.environ.get("DATASET_POSTGRES_PORT", "5432"),
    dbname=os.environ.get("DATASET_POSTGRES_DB", "heart_disease"),
    user=os.environ.get("DATASET_POSTGRES_USER", "dataset_user"),
    password=os.environ.get("DATASET_POSTGRES_PASSWORD", "dataset_pass"),
)
print("Connected database:", conn.get_dsn_parameters()["dbname"])


Connected database: heart_disease


In [2]:
expected_tables = {"heart_disease_raw", "heart_disease_processed"}
with conn.cursor() as cur:
    cur.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = %s", ("public",))
    existing_tables = {row[0] for row in cur.fetchall()}
missing = expected_tables - existing_tables
if missing:
    raise RuntimeError(f"Missing tables {sorted(missing)}. Run notebook 02 first.")

for table in sorted(expected_tables):
    with conn.cursor() as cur:
        cur.execute(f"SELECT COUNT(*) FROM public.{table}")
        print(f"{table}: {cur.fetchone()[0]} rows")


heart_disease_processed: 297 rows
heart_disease_raw: 303 rows


In [3]:
with conn.cursor() as cur:
    cur.execute("SELECT * FROM heart_disease_raw LIMIT 5")
    raw_preview = pd.DataFrame(cur.fetchall(), columns=[desc.name for desc in cur.description])
    cur.execute("SELECT * FROM heart_disease_processed LIMIT 5")
    processed_preview = pd.DataFrame(cur.fetchall(), columns=[desc.name for desc in cur.description])
print("Raw data preview")
display(raw_preview)
print("Processed data preview")
display(processed_preview)
conn.close()


Raw data preview


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


Processed data preview


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0
